# Phishing Website Detection Using URL-Based Features
**CS746 | Perspectives on Data Science | Fall 2026**

This notebook implements the project plan using only information parsed from each raw URL. It does not visit or download webpages.

**Important label convention:** UCI documents `label = 1` as legitimate and `label = 0` as phishing. This notebook converts the target to `1 = phishing` so that phishing-class precision, recall, and F1 are reported consistently.

The dataset contains precomputed webpage and URL features, but this project deliberately uses the raw `URL` field to create its own URL-only predictors. `FILENAME`, `Domain`, `TLD`, and the precomputed feature columns are not used as predictors.

In [1]:
!pip -q install ucimlrepo tldextract

## 1. Imports and reproducibility

In [4]:
import re
import ipaddress
import numpy as np
import pandas as pd
import tldextract

from urllib.parse import urlsplit
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

## 2. Load the official UCI dataset and inspect it

In [13]:
dataset = fetch_ucirepo(id=967)
raw = dataset.data.features.copy()
target = dataset.data.targets.copy()

print("Feature-table shape:", raw.shape)
print("Target-table shape:", target.shape)
print("Columns:", raw.columns.tolist())
display(raw[["URL"]].head())
display(target.head())

# Keep only the raw URL and official label. The URL column is the only source
# used to construct predictors; the other precomputed columns are excluded.
df = pd.DataFrame({
    "url": raw["URL"].astype("string"),
    "uci_label": target.iloc[:, 0]
})

print("\\nOriginal UCI label counts (1=legitimate, 0=phishing):")
display(df["uci_label"].value_counts(dropna=False).rename("count").to_frame())

Feature-table shape: (235795, 54)
Target-table shape: (235795, 1)
Columns: ['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


,URL
0,https://www.southbankmosaics.com
1,https://www.uni-mainz.de
2,https://www.voicefmradio.co.uk
3,https://www.sfnmjournal.com
4,https://www.rewildingargentina.org


,label
0,1
1,1
2,1
3,1
4,1


\nOriginal UCI label counts (1=legitimate, 0=phishing):


,count
uci_label,
1,134850
0,100945


## 3. Clean URLs and create the phishing target

In [14]:
df["url"] = df["url"].str.strip()
df = df.dropna(subset=["url", "uci_label"]).copy()
df = df[df["url"].ne("")].copy()
df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

# UCI: 1 = legitimate, 0 = phishing. Project target: 1 = phishing.
df["phishing"] = (df["uci_label"].astype(int) == 0).astype(int)

def parse_url(value):
    value = str(value).strip()
    candidate = value if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", value) else "http://" + value
    try:
        parsed = urlsplit(candidate)
        host = (parsed.hostname or "").lower().rstrip(".")
        if not host:
            return None, None
        return parsed, host
    except Exception:
        return None, None

parsed = df["url"].map(parse_url)
df["parsed"] = parsed.map(lambda x: x[0])
df["host"] = parsed.map(lambda x: x[1])
df = df.dropna(subset=["parsed", "host"]).reset_index(drop=True)

print("Rows after removing missing, empty, exact-duplicate, and unparsable URLs:", len(df))
print("Phishing class (1) and legitimate class (0) counts:")
display(df["phishing"].value_counts().sort_index().rename(index={0:"Legitimate", 1:"Phishing"}).to_frame("count"))
display((df["phishing"].value_counts(normalize=True).sort_index()*100).rename(index={0:"Legitimate", 1:"Phishing"}).round(2).to_frame("percent"))

Rows after removing missing, empty, exact-duplicate, and unparsable URLs: 235370
Phishing class (1) and legitimate class (0) counts:


,count
phishing,
Legitimate,134850
Phishing,100520


,percent
phishing,
Legitimate,57.29
Phishing,42.71


## 4. Extract URL-only features

In [15]:
def extract_features(url):
    parsed, host = parse_url(url)
    if parsed is None:
        return None

    path = parsed.path or ""
    query = parsed.query or ""
    full = str(url)
    try:
        ipaddress.ip_address(host)
        host_is_ip = 1
    except ValueError:
        host_is_ip = 0

    # tldextract separates subdomains from the registered domain.
    ext = tldextract.extract(host)
    subdomain_count = len([p for p in ext.subdomain.split(".") if p]) if ext.subdomain else 0

    return {
        "url_length": len(full),
        "hostname_length": len(host),
        "path_length": len(path),
        "digit_count": sum(ch.isdigit() for ch in full),
        "dot_count": full.count("."),
        "hyphen_count": full.count("-"),
        "subdomain_count": subdomain_count,
        "query_parameter_count": len([p for p in query.split("&") if p]) if query else 0,
        "uses_https": int(parsed.scheme.lower() == "https"),
        "has_at_symbol": int("@" in full),
        "hostname_is_ip": host_is_ip,
    }

feature_rows = [extract_features(u) for u in df["url"]]
X = pd.DataFrame(feature_rows, index=df.index)
valid = X.notna().all(axis=1)
df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)
y = df["phishing"].astype(int)

# Group by registered domain to reduce leakage between partitions.
def registered_domain(host):
    try:
        ipaddress.ip_address(host)
        return host
    except ValueError:
        ext = tldextract.extract(host)
        if ext.domain and ext.suffix:
            return ext.domain + "." + ext.suffix
        return host

groups = df["host"].map(registered_domain)
print("Feature matrix:", X.shape)
print("Unique registered-domain groups:", groups.nunique())
display(X.head())

Feature matrix: (235370, 11)
Unique registered-domain groups: 175509


,url_length,hostname_length,path_length,digit_count,dot_count,hyphen_count,subdomain_count,query_parameter_count,uses_https,has_at_symbol,hostname_is_ip
0,32,24,0,0,2,0,1,0,1,0,0
1,24,16,0,0,2,1,1,0,1,0,0
2,30,22,0,0,3,0,1,0,1,0,0
3,27,19,0,0,2,0,1,0,1,0,0
4,34,26,0,0,2,0,1,0,1,0,0


## 5. Domain-grouped train/validation/test split

In [16]:
# StratifiedGroupKFold keeps every registered domain in one partition.
# Select the fold whose test size is closest to 20% and whose phishing
# proportion is closest to the full dataset proportion.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
overall_rate = y.mean()
best = None
for train_idx, test_idx in outer.split(X, y, groups):
    score = abs(len(test_idx)/len(y) - 0.20) + abs(y.iloc[test_idx].mean() - overall_rate)
    if best is None or score < best[0]:
        best = (score, train_idx, test_idx)

dev_idx, test_idx = best[1], best[2]
X_dev, X_test = X.iloc[dev_idx], X.iloc[test_idx]
y_dev, y_test = y.iloc[dev_idx], y.iloc[test_idx]
g_dev, g_test = groups.iloc[dev_idx], groups.iloc[test_idx]

inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)
best_inner = None
for tr_rel, val_rel in inner.split(X_dev, y_dev, g_dev):
    score = abs(len(val_rel)/len(y_dev) - 0.20) + abs(y_dev.iloc[val_rel].mean() - y_dev.mean())
    if best_inner is None or score < best_inner[0]:
        best_inner = (score, tr_rel, val_rel)

tr_idx, val_idx = best_inner[1], best_inner[2]
X_train, X_val = X_dev.iloc[tr_idx], X_dev.iloc[val_idx]
y_train, y_val = y_dev.iloc[tr_idx], y_dev.iloc[val_idx]
g_train, g_val = g_dev.iloc[tr_idx], g_dev.iloc[val_idx]

assert set(g_train).isdisjoint(set(g_val))
assert set(g_train).isdisjoint(set(g_test))
assert set(g_val).isdisjoint(set(g_test))

split_summary = pd.DataFrame({
    "partition": ["Training", "Validation", "Final test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "phishing_percent": [100*y_train.mean(), 100*y_val.mean(), 100*y_test.mean()],
    "unique_domains": [g_train.nunique(), g_val.nunique(), g_test.nunique()]
})
display(split_summary.round(2))

,partition,rows,phishing_percent,unique_domains
0,Training,148066,41.62,112336
1,Validation,38538,44.42,28076
2,Final test,48766,44.66,35097


## 6. Train logistic regression and random forest

In [19]:
# All preprocessing is fitted inside each pipeline using training data only.
logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced",
                                 random_state=RANDOM_STATE))
])

forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        n_jobs=-1, random_state=RANDOM_STATE
    ))
])

models = {
    "Logistic Regression": logistic,
    "Random Forest": forest
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained.")

Logistic Regression trained.
Random Forest trained.


## 7. Compare models on validation data

In [20]:
def evaluate(model, X_part, y_part):
    pred = model.predict(X_part)
    tn, fp, fn, tp = confusion_matrix(y_part, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_part, pred),
        "precision_phishing": precision_score(y_part, pred, zero_division=0),
        "recall_phishing": recall_score(y_part, pred, zero_division=0),
        "f1_phishing": f1_score(y_part, pred, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0.0,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

validation_results = pd.DataFrame({
    name: evaluate(model, X_val, y_val) for name, model in models.items()
}).T
display(validation_results.round(4))

,accuracy,precision_phishing,recall_phishing,f1_phishing,false_positive_rate,TN,FP,FN,TP
Logistic Regression,0.9756,0.9969,0.9481,0.9719,0.0024,21369.0,51.0,888.0,16230.0
Random Forest,0.9909,0.9993,0.9801,0.9896,0.0006,21408.0,12.0,340.0,16778.0


## 8. Permutation feature importance (validation set)

In [21]:
# Use phishing-class F1 as the scoring measure.
for name, model in models.items():
    result = permutation_importance(
        model, X_val, y_val, scoring="f1", n_repeats=5,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    importance = pd.DataFrame({
        "feature": X.columns,
        "mean_f1_decrease": result.importances_mean,
        "std": result.importances_std
    }).sort_values("mean_f1_decrease", ascending=False)
    print(name)
    display(importance)

Logistic Regression


,feature,mean_f1_decrease,std
0,url_length,0.280953,0.002569
8,uses_https,0.278992,0.000857
1,hostname_length,0.129056,0.001016
2,path_length,0.118634,0.001747
3,digit_count,0.084127,0.000833
5,hyphen_count,0.018696,0.000290
7,query_parameter_count,0.006262,0.000245
9,has_at_symbol,0.003101,0.000203
6,subdomain_count,0.002902,0.000113
10,hostname_is_ip,0.000333,0.000074


Random Forest


,feature,mean_f1_decrease,std
8,uses_https,0.260778,0.000964
2,path_length,0.199053,0.001456
3,digit_count,0.039874,0.000485
6,subdomain_count,0.027502,0.000891
0,url_length,0.008546,0.000520
1,hostname_length,0.001658,0.000106
4,dot_count,0.001074,0.000119
5,hyphen_count,0.000884,0.000139
7,query_parameter_count,0.000012,0.000014
9,has_at_symbol,0.000000,0.000000


## 9. Final evaluation on the untouched test set

In [22]:
# Select the model using validation phishing-class F1 only.
best_name = validation_results["f1_phishing"].idxmax()
best_model = models[best_name]
print("Selected using validation F1:", best_name)

test_results = evaluate(best_model, X_test, y_test)
display(pd.DataFrame([test_results], index=[best_name]).round(4))
print(classification_report(
    y_test, best_model.predict(X_test),
    labels=[0, 1], target_names=["Legitimate", "Phishing"],
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted; order 0=legitimate, 1=phishing):")
print(confusion_matrix(y_test, best_model.predict(X_test), labels=[0, 1]))

Selected using validation F1: Random Forest


,accuracy,precision_phishing,recall_phishing,f1_phishing,false_positive_rate,TN,FP,FN,TP
Random Forest,0.9965,0.9994,0.9928,0.9961,0.0005,26975,13,157,21621


              precision    recall  f1-score   support

  Legitimate       0.99      1.00      1.00     26988
    Phishing       1.00      0.99      1.00     21778

    accuracy                           1.00     48766
   macro avg       1.00      1.00      1.00     48766
weighted avg       1.00      1.00      1.00     48766

Confusion matrix (rows=true, columns=predicted; order 0=legitimate, 1=phishing):
[[26975    13]
 [  157 21621]]


## 10. Notes and limitations

- The target is coded so that `1` means phishing, consistent with the reported phishing-class metrics.
- Only URL-derived structural features are predictors; webpage content and the dataset's precomputed features are excluded.
- Registered domains are grouped so the same domain does not appear in more than one partition. The stratified group split is approximate; inspect the displayed class percentages and partition sizes.
- The final test set is used only after model selection on validation data.
- Results describe this benchmark dataset and do not guarantee performance on future phishing campaigns.